# 🔥 PyTorch: the graph it records

*Sections 3.1 and 15.1.1 of the notes.*

In Chapter 3 we drew the **computational graph** of a function by hand: input
nodes, operation nodes, and edges from each node to the nodes it is computed
from. Then the **reverse sweep** walked that graph backwards and produced all
the partial derivatives at once.

PyTorch builds the graph **for you, while the program runs**. 🏗️ A
**tensor** is an array that can carry a derivative. Every operation on tensors
creates a new node that remembers

* which function produced it, and
* which tensors it came from.

This notebook builds the graph of Example 3.2, opens it up, runs the reverse
sweep, meets the two traps of this design, and ends with the second derivative
that a physics-informed network needs (Chapter 5).

## ⚙️ Setup

We compute in double precision, so that the checks against formulas agree to
many digits.

🖼️ To draw graphs we use **torchviz**. It needs the program **Graphviz**,
which is installed outside Python: `brew install graphviz` on macOS,
`sudo apt install graphviz` on Ubuntu, or the installer from graphviz.org on
Windows.

In [ ]:
import torch
import torch.nn as nn
from torchviz import make_dot

torch.set_default_dtype(torch.float64)
print("torch", torch.__version__)

## 1. 🧱 Building the graph of Example 3.2

The function of Example 3.2 is

$$ f(x_1, x_2) = \sin(x_1 x_2) + x_2^3 , $$

evaluated at $x_1 = 2$, $x_2 = 3$. Its graph (Figure 3.2) has two input nodes
and four operation nodes:

| node | value |
|---|---|
| $v_1$, $v_2$ | the inputs $x_1$, $x_2$ |
| $v_3$ | $v_1 v_2$ |
| $v_4$ | $v_2^3$ |
| $v_5$ | $\sin v_3$ |
| $v_6$ | $v_4 + v_5$, the output |

`requires_grad=True` marks a tensor as an **input node**: a quantity we want
derivatives with respect to. We write one line per node.

In [ ]:
x1 = torch.tensor(2.0, requires_grad=True)  # input node v1
x2 = torch.tensor(3.0, requires_grad=True)  # input node v2

v3 = x1 * x2
v4 = x2**3
v5 = torch.sin(v3)
v6 = v4 + v5  # the output

print("f(2, 3) =", v6.item())

## 2. 🔍 Opening the graph

Every part of Definition 3.1 has a place in PyTorch (Table 15.1 of the notes):

| computational graph | PyTorch |
|---|---|
| value $v_k$ | the numbers held by the tensor |
| input node | a **leaf**: created by us with `requires_grad=True` |
| function $\varphi_k$ of an operation node | `grad_fn` |
| parents $P(k)$ | `grad_fn.next_functions` |
| values needed for the partial derivatives | tensors saved by `grad_fn` |
| derivative of the output with respect to $x_\ell$ | `.grad` of the leaf |

The function `show` below starts at the output and follows `next_functions`
from each node to its parents, printing the graph **backwards**, one level of
indentation per step.

* `type(fn).__name__` is the name of the class of the node, such as
  `MulBackward0` for a product.
* `fn.next_functions` holds one pair per parent: the parent node, and an index
  that PyTorch uses internally. We use only the node, and write `_` for the
  index. A parent that needs no derivative is `None`, and is skipped.

In [ ]:
def show(fn, depth):
    # Print the node fn and, indented below it, the nodes it was computed from.
    print("    " * depth + type(fn).__name__)
    for parent, _ in fn.next_functions:
        if parent is not None:
            show(parent, depth + 1)


print("is x1 a leaf?", x1.is_leaf, "| its grad_fn:", x1.grad_fn)
print("is v6 a leaf?", v6.is_leaf)
print()
show(v6.grad_fn, 0)

💡 **What happened?** Read the printout from the top: the output came from an
**addition** (`AddBackward0`), whose parents are the **power** $v_4$ and the
**sine** $v_5$. The sine came from the **product** $v_3$, and the product and
the power lead to the inputs. The input $x_2$ appears twice, because it feeds
two nodes. An input node appears as `AccumulateGrad`: the
step of the reverse sweep that adds a derivative into the `.grad` of a leaf.

The reverse sweep needs the partial derivatives of each operation, so each node
keeps the values those derivatives need. The next cell prints these saved values.
PyTorch names the arguments of an operation `self` (the first) and `other` (the
second), so the product node saves its factors as `_saved_self` ($v_1$) and
`_saved_other` ($v_2$).

🤔 *Guess: which numbers does the product node $v_3 = v_1 v_2$ keep? And the
sine node $v_5 = \sin v_3$?*

In [ ]:
product = v3.grad_fn
print(
    "v3 = v1 * v2 stores v1 =",
    product._saved_self.item(),
    "and v2 =",
    product._saved_other.item(),
)

sine = v5.grad_fn
print("v5 = sin(v3) stores v3 =", sine._saved_self.item())
print("is the stored tensor v3 itself?", sine._saved_self is v3)

💡 The product keeps **both** factors, because
$\partial(v_1 v_2)/\partial v_1 = v_2$ and $\partial(v_1 v_2)/\partial v_2 = v_1$.
The sine keeps its argument, because its derivative is $\cos v_3$.

👉 These saved values are the stored values of Section 3.4: the **memory** of
a recorded graph grows with the number of operations.

### 🖼️ Drawing the graph

`make_dot` from torchviz draws the graph that `show` printed. Each tensor box
shows a name and the shape of the tensor, `()` for a single number. The names
come from the dictionary `params`; a tensor missing from it gets no name. The
colors:

* 🟦 **blue**: the input nodes (leaves);
* ⬜ **gray**: the operation nodes, one `grad_fn` each, and the
  `AccumulateGrad` steps that write into `.grad`;
* 🟩 **green**: the output.

The arrows point in the direction of the forward computation. Compare with
Figure 3.2 of the notes.

In [ ]:
make_dot(v6, params={"x1": x1, "x2": x2, "v6": v6})

## 3. ⏪ The reverse sweep

**What it computes.** The reverse sweep attaches to every node $k$ an
**adjoint variable** $\bar v_k$: the derivative of the output with respect to
$v_k$, through the part of the computation that comes after node $k$
(Definition 3.6). It starts at the output with the seed $\bar v_6 = \bar f
= 1$ and walks the graph **backwards**. Each node collects one term from each
of its children $j$ (Proposition 3.7):

$$ \bar v_k = \sum_{j \in C(k)} \bar v_j \, \frac{\partial \varphi_j}{\partial v_k} . $$

For Example 3.2 the sweep is (Example 3.8):

| node | adjoint variable | at $x_1 = 2$, $x_2 = 3$ |
|---|---|---|
| $v_6 = v_4 + v_5$ | $\bar v_6 = 1$ | $1$ |
| $v_5 = \sin v_3$ | $\bar v_5 = \bar v_6$ | $1$ |
| $v_4 = v_2^3$ | $\bar v_4 = \bar v_6$ | $1$ |
| $v_3 = v_1 v_2$ | $\bar v_3 = (\cos v_3)\,\bar v_5$ | $\cos 6$ |
| $v_2 = x_2$ | $\bar v_2 = v_1\,\bar v_3 + 3v_2^2\,\bar v_4$ | $2\cos 6 + 27$ |
| $v_1 = x_1$ | $\bar v_1 = v_2\,\bar v_3$ | $3\cos 6$ |

For an input node, the remaining computation is $f$ itself, so the adjoint
variable is the inner product of the seed with the derivative of $f$
(equation (3.1)):

$$ \bar v_k = \bar{\boldsymbol f}^\top \frac{\partial \boldsymbol f}{\partial x_k}(\boldsymbol x), \qquad k = 1, 2 . $$

Here $f$ is scalar and the seed is $\bar f = 1$, so the adjoint variables of
the inputs are the gradient:
$(\bar v_1, \bar v_2) = \nabla f(x_1, x_2)$.
Node $2$ has two children, $v_3$ and $v_4$, so its adjoint variable has two
terms. Every partial derivative in the table is evaluated at the **saved
values** of Section 2: $v_1$ and $v_2$ for the product, $v_3$ for the sine.

The next cell runs the sweep by hand. It computes inside
`with torch.no_grad():`, which tells PyTorch not to record these operations:
they are our check, not part of the graph of $f$.

In [ ]:
with torch.no_grad():  # the hand computation is not part of the graph
    bar_v6 = torch.tensor(1.0)  # the seed
    bar_v5 = bar_v6
    bar_v4 = bar_v6
    bar_v3 = torch.cos(v3) * bar_v5
    bar_v2 = x1 * bar_v3 + 3 * x2**2 * bar_v4
    bar_v1 = x2 * bar_v3

print("bar v6 =", bar_v6.item())
print("bar v5 =", bar_v5.item())
print("bar v4 =", bar_v4.item())
print("bar v3 =", bar_v3.item())
print("bar v2 =", bar_v2.item())
print("bar v1 =", bar_v1.item())

Now PyTorch runs the same sweep with `backward()`. It writes $\bar v_1$ and
$\bar v_2$ into the `.grad` of the leaves.

In [ ]:
v6.backward()  # the reverse sweep, with the seed 1

print(f"df/dx1  PyTorch {x1.grad.item():.6f}   bar v1 by hand {bar_v1.item():.6f}")
print(f"df/dx2  PyTorch {x2.grad.item():.6f}   bar v2 by hand {bar_v2.item():.6f}")

## 4. ➕ Trap 1: gradients add up

We compute $f$ again, from the same leaves, and call `backward()` a second time.

🤔 *Guess: what is `x1.grad` now?*

In [ ]:
y = torch.sin(x1 * x2) + x2**3
y.backward()
print(f"after the second sweep: df/dx1 = {x1.grad.item():.6f}")

x1.grad = None  # clear the stored gradients
x2.grad = None
y = torch.sin(x1 * x2) + x2**3
y.backward()
print(f"after clearing first:   df/dx1 = {x1.grad.item():.6f}")

💡 **What happened?** The second sweep **added** its result to the stored one,
so the value doubled. `backward` returns nothing; it writes into the leaves,
and what is written stays there.

This is on purpose: the gradient of a sum is the sum of the gradients, so a
batch too large for memory can be processed in pieces, one `backward()` per
piece, and `.grad` ends up holding the gradient of the whole batch. But it is
why every training loop clears the gradients before each step, and forgetting
it is a classic bug. 🐛

Three ways clear the stored gradients:

* `x.grad = None`, as in the cell above: the memory is freed, and the next
  `backward()` writes a fresh gradient;
* `x.grad.zero_()`: fills the gradient with zeros in place; the memory stays,
  and the next `backward()` adds to the zeros;
* `optimizer.zero_grad()` or `model.zero_grad()`: clears the gradients of all
  parameters at once. Since PyTorch 2.0 they set them to `None`;
  `set_to_none=False` zeros them instead.

## 5. 🗑️ Trap 2: the graph is freed after the sweep

The saved tensors take memory, so `backward` frees the graph as soon as it is
done.

🤔 *Guess: what happens if we call `backward()` twice on the **same** output,
without computing it again?*

In [ ]:
y = torch.sin(x1 * x2) + x2**3
y.backward()
try:
    y.backward()
except RuntimeError as err:
    print("RuntimeError:", err)

x1.grad = None
y = torch.sin(x1 * x2) + x2**3
y.backward(retain_graph=True)  # keep the graph for one more sweep
y.backward()
print(f"\nwith retain_graph=True: df/dx1 = {x1.grad.item():.6f} (two sweeps added)")

💡 The second sweep finds the saved tensors gone. `retain_graph=True` keeps
them, at the price of the memory.

## 6. 🔁 Second derivatives

The reverse sweep is itself a computation. With `create_graph=True`, PyTorch
**records the sweep** as a new graph, which can be differentiated again. For
$u(x) = \sin(x^2)$,

$$ u'(x) = 2x\cos(x^2), \qquad u''(x) = 2\cos(x^2) - 4x^2\sin(x^2). $$

`torch.autograd.grad(output, input)` returns the derivative instead of writing
it into `.grad`.

🤔 *Guess: what happens if we differentiate $u'$ again without
`create_graph=True`?*

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
u = torch.sin(x**2)

(du,) = torch.autograd.grad(u, x)  # the sweep is not recorded
print("du =", du, "| its grad_fn:", du.grad_fn)

try:
    torch.autograd.grad(du, x)
except RuntimeError as error:
    print("RuntimeError:", error)

💡 **What happened?** Without `create_graph=True`, the sweep returns $u'(1)$ as
a plain number: `du` has no `grad_fn`, so there is no graph from `du` back to
`x`, and PyTorch has nothing to differentiate.

With `create_graph=True`, the operations of the sweep are recorded as they run.
The result `du` then has a `grad_fn`, and a second sweep through it gives $u''$.

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
u = torch.sin(x**2)

(du,) = torch.autograd.grad(u, x, create_graph=True)  # the sweep is recorded
print("du =", du, "| its grad_fn:", du.grad_fn)

(d2u,) = torch.autograd.grad(du, x)  # differentiate the sweep

with torch.no_grad():  # the formulas, at x = 1
    du_exact = 2 * x * torch.cos(x**2)
    d2u_exact = 2 * torch.cos(x**2) - 4 * x**2 * torch.sin(x**2)

print("u'(1):  PyTorch", du.item(), "  formula", du_exact.item())
print("u''(1): PyTorch", d2u.item(), "  formula", d2u_exact.item())

## 7. 🌡️ The residual of a physics-informed network

The residual of Chapter 5, equation (5.2), contains the second derivative
$u_{xx}$ of a **network** $u(x)$ at many points $x_1, \dots, x_N$. These points
discretize the space domain $[-1, 1]$; below they are $N = 5$ equally spaced
points. The two calls of Section 6 give $u_{xx}$ at all $N$ points, with one
sweep each:

* the network maps the column of points $(x_1, \dots, x_N)$ to the column of
  values $(u(x_1), \dots, u(x_N))$;
* the output is a vector, so `torch.autograd.grad` needs a seed
  $\bar{\boldsymbol f}$ (Definition 3.6). The seed `torch.ones_like(u)` is the
  vector of ones: it asks for the derivative of $\sum_i u(x_i)$ with respect to
  each $x_j$;
* the value $u(x_i)$ depends only on its own point $x_i$. The derivative of the
  sum with respect to $x_j$ is therefore $u'(x_j)$, and one sweep gives $u'$ at
  every point;
* the second call does the same for $u'$ and gives $u''$ at every point.

To check the result we compare $u_{xx}$ with the second difference of
Chapter 2, $(u(x+h) - 2u(x) + u(x-h))/h^2$.

What the cell uses:

* `nn.Sequential(...)` chains the layers into one network: a linear layer from
  1 to 16 numbers, $\tanh$, and a linear layer from 16 to 1.
  `torch.manual_seed(0)` fixes the random initial weights, so every run gives
  the same network.
* `torch.linspace(-1, 1, 5)` gives the five points. A layer takes a **batch**
  of points, an array of shape `(number of points, 1)`, and
  `.reshape(-1, 1)` makes the points a column of that shape; `.flatten()`
  turns such a column back into a row for printing (`-1` means: as
  many rows as needed). `.requires_grad_(True)` marks the points as an input
  node, as `requires_grad=True` did in Section 1.
* The second difference is computed under `torch.no_grad()`: it is a check, not
  part of the graph.
* The network has no time argument, so $u_t = 0$, and the residual (5.2) is
  $r = -(D u_{xx} + u - u^3)$.

🤔 *Guess: how close do the two agree, with $h = 10^{-3}$?*

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))

points = torch.linspace(-1, 1, 5)
x = points.reshape(-1, 1).requires_grad_(True)
u = net(x)
(u_x,) = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)
(u_xx,) = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)

h = 1e-3
with torch.no_grad():
    second_difference = (net(x + h) - 2 * net(x) + net(x - h)) / h**2

largest_gap = (u_xx - second_difference).abs().max()

print("x:                ", points)
print("u_xx:             ", u_xx.flatten())
print("second difference:", second_difference.flatten())
print("largest |u_xx - second difference|:", largest_gap.item())

diffusion = 2e-5  # the diffusion coefficient D of Allen-Cahn
residual = -(diffusion * u_xx + u - u**3)  # u_t = 0
print("\nresidual shape:", tuple(residual.shape))

💡 **What happened?** The two differ by at most $5\cdot10^{-8}$, the error of
the second difference, of order $h^2$ (Proposition 2.12). The reverse sweep
gives $u_{xx}$ **exactly**, up to rounding, with no step size to choose.

`u_xx` carries a `grad_fn`. Because we passed `create_graph=True` the second
time too, the residual is itself part of a graph: a loss built from it can be differentiated with respect
to the weights of the network, which is how a physics-informed network is
trained.

### 🖼️ The graph of a network, and of its second derivative

First the graph of the network itself. Its leaves are the input `x` and the
four parameter arrays. `dict(net.named_parameters())` is a dictionary from the
name of each parameter array to the array, and we add `x` to it. The operator
`|` joins two dictionaries; it adds the name of the output. `AddmmBackward0` is
one layer, $\boldsymbol W\boldsymbol h + \boldsymbol b$, and `TBackward0`
transposes a weight matrix.

In [ ]:
leaves = dict(net.named_parameters())
leaves["x"] = x
make_dot(u, params=leaves | {"u": u})

Now the graph of $u_{xx}$. It contains the **recorded reverse sweeps**: nodes
such as `TanhBackwardBackward0` are the derivative of the derivative of
$\tanh$.

🤔 *Guess: one of the five leaves of the network does not appear in the graph
of $u_{xx}$. Which one, and why?*

In [ ]:
make_dot(u_xx, params=leaves | {"u_xx": u_xx})

💡 The bias `2.bias` of the last layer is gone. It adds a constant to $u$, and
the second derivative of a constant is zero, so $u_{xx}$ does not depend on it.
A loss built from $u_{xx}$ alone would give this bias no gradient.

## 🎯 Summary

| You want | PyTorch |
|---|---|
| mark an input node | `torch.tensor(..., requires_grad=True)` |
| the graph | built automatically, node by node, as the code runs |
| all first derivatives | `y.backward()`, then read `.grad` |
| start again | set `.grad` to `None` (or `optimizer.zero_grad()`) |
| sweep the same graph twice | `retain_graph=True` |
| a second derivative | `torch.autograd.grad(..., create_graph=True)`, twice |
| a picture of the graph | `make_dot(y, params=...)` from torchviz |

Remember the two traps: **gradients add up**, and **the graph is freed** after
the sweep.

## 🧪 Try it yourself

1. In Section 2, print `v6.grad_fn.next_functions` directly. Which entry is
   the power and which the sine? Compare with the order of the terms in `v6`.
2. In Section 6, compute $u'''(1)$ by adding a third call. What must the
   second call get that it did not have before?
3. In Section 7, change $h$ to $10^{-2}$ and to $10^{-6}$. Why does the
   agreement get worse in **both** directions? (Hint: in double precision,
   $u(x+h) - 2u(x) + u(x-h)$ loses digits when $h$ is small.)